# Hypoxia Classification from Single-Cell RNA-seq, Combined Model Report

**Group: Gradient Descenders.**

This notebook unifies the whole group's work into a single, uniform pipeline. From one shared data foundation we classify single cells as **hypoxia** (about 1 percent oxygen) versus **normoxia** (about 21 percent oxygen) across two breast-cancer cell lines (MCF7, HCC1806) and two single-cell RNA-seq technologies (SmartSeq, DropSeq).

Every model that was explored across the separate notebooks appears here on a common footing: LASSO, Linear SVM, L2 Logistic Regression, K Nearest Neighbours, an MLP, a 1D CNN, Random Forest, Extra Trees, XGBoost, an Autoencoder with a denoising variant, and Graph Neural Networks (GCN and GraphSAGE), alongside PCA, t-SNE and K-Means. We then study cross-subset generalisation, feature importance and hypoxia biology, a pooled leave-one-group-out model, and the statistical independence of the two cell lines.

**Execution.** The notebook is written to run in WSL on an NVIDIA GPU through RAPIDS cuML, and to fall back cleanly to CPU on plain Windows. Every GPU or special-purpose library is imported behind a flag, so a missing library skips its section with a message rather than stopping the run. The underlying expression matrices are confidential and never leave the local machine; only aggregate metrics, gene names and figures appear here.

## 1. Environment setup

We import everything once and probe for the optional accelerators and libraries. `cuml.accel.install()` transparently moves scikit-learn estimators onto the GPU when available. PyTorch, XGBoost, torch-geometric, networkx, SHAP, dcor and scikit-bio are each optional: the boolean flags below let later sections run when the library is present and skip gracefully when it is not. A single global `SEED` makes the run reproducible.

In [ ]:
try:
    import cuml.accel
    cuml.accel.install()
    CUML_ACCEL = True
    _CUML_ERR = None
except Exception as _e:
    CUML_ACCEL = False
    _CUML_ERR = repr(_e)

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (StratifiedKFold, cross_validate, cross_val_score,
                                      GridSearchCV, train_test_split, LeaveOneGroupOut)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier, kneighbors_graph
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, adjusted_rand_score)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.calibration import CalibratedClassifierCV

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

try:
    import joblib
    joblib.parallel_backend('threading')
except Exception:
    pass

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
    torch.manual_seed(SEED)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(SEED)
except Exception:
    TORCH_OK = False
    DEVICE = None

try:
    from xgboost import XGBClassifier
    XGB_OK = True
except Exception:
    XGB_OK = False

try:
    from torch_geometric.data import Data
    from torch_geometric.nn import GCNConv, SAGEConv
    from torch_geometric.utils import to_networkx
    PYG_OK = TORCH_OK
except Exception:
    PYG_OK = False

try:
    import networkx as nx
    NX_OK = True
except Exception:
    NX_OK = False

try:
    import shap
    SHAP_OK = True
except Exception:
    SHAP_OK = False

try:
    import dcor
    DCOR_OK = True
except Exception:
    DCOR_OK = False

try:
    from skbio.stats.distance import DistanceMatrix, mantel
    SKBIO_OK = True
except Exception:
    SKBIO_OK = False

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
warnings.filterwarnings('ignore')

print('cuML accelerator :', CUML_ACCEL)
print('PyTorch          :', TORCH_OK, '| device:', DEVICE)
print('XGBoost          :', XGB_OK)
print('torch_geometric  :', PYG_OK, '| networkx:', NX_OK)
print('shap             :', SHAP_OK, '| dcor:', DCOR_OK, '| skbio:', SKBIO_OK)

## 2. Loading the data

We read every expression matrix into memory as a genes by cells DataFrame and build one metadata table per cell. `DATA_ROOT` is auto-detected so the same notebook runs from a Windows path or a WSL mount. Only SmartSeq ships the full unfiltered then filtered then normalised ladder; DropSeq ships only the normalised version. Importantly the files named normalised are still raw integer counts restricted to the top 3000 most variable genes; per-gene standardisation happens at model-fitting time.

In [ ]:
_CANDIDATES = [
    Path(r'C:/Users/ricca/OneDrive/universita/4_semestre/AI_lab/AILab2026/AILab2026'),
    Path('/mnt/c/Users/ricca/OneDrive/universita/4_semestre/AI_lab/AILab2026/AILab2026'),
]
DATA_ROOT = next((p for p in _CANDIDATES if p.exists()), _CANDIDATES[0])
SMART = DATA_ROOT / 'SmartSeq'
DROP = DATA_ROOT / 'DropSeq'
OUT_DIR = Path('../outputs')
OUT_DIR.mkdir(exist_ok=True)

assert SMART.exists() and DROP.exists(), f'Data folders not found, check DATA_ROOT (tried: {[str(p) for p in _CANDIDATES]}).'

def load_expr(path):
    df = pd.read_csv(path, sep=r'\s+', engine='python', index_col=0)
    df.index = df.index.astype(str).str.replace('"', '', regex=False)
    df.columns = df.columns.astype(str).str.replace('"', '', regex=False)
    return df

print(f'OK, using DATA_ROOT = {DATA_ROOT}')

In [ ]:
matrices = {}

matrices[('SmartSeq', 'MCF7', 'unfiltered')]  = load_expr(SMART / 'MCF7_SmartS_Unfiltered_Data.txt')
matrices[('SmartSeq', 'MCF7', 'filtered')]    = load_expr(SMART / 'MCF7_SmartS_Filtered_Data.txt')
matrices[('SmartSeq', 'MCF7', 'normalised')]  = load_expr(SMART / 'MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt')
matrices[('SmartSeq', 'HCC1806', 'unfiltered')] = load_expr(SMART / 'HCC1806_SmartS_Unfiltered_Data.txt')
matrices[('SmartSeq', 'HCC1806', 'filtered')]   = load_expr(SMART / 'HCC1806_SmartS_Filtered_Data.txt')
matrices[('SmartSeq', 'HCC1806', 'normalised')] = load_expr(SMART / 'HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt')

matrices[('DropSeq', 'MCF7', 'normalised')]    = load_expr(DROP / 'MCF7_Filtered_Normalised_3000_Data_train.txt')
matrices[('DropSeq', 'HCC1806', 'normalised')] = load_expr(DROP / 'HCC1806_Filtered_Normalised_3000_Data_train.txt')

summary_rows = []
for (tech, line, stage), df in matrices.items():
    summary_rows.append({'technology': tech, 'cell_line': line, 'stage': stage,
                         'n_genes': df.shape[0], 'n_cells': df.shape[1]})
summary_df = pd.DataFrame(summary_rows).sort_values(['technology', 'cell_line', 'stage']).reset_index(drop=True)
summary_df

### 2.1 Extracting labels

Condition labels live in the cell names themselves: SmartSeq names contain `_Hypo_` or `_Norm_`, DropSeq names end in `_Hypoxia` or `_Normoxia`. We normalise both to a single two-class label and also keep a per-cell log1p copy of every matrix for plotting and for the representation-learning section.

In [ ]:
def extract_label(cell_name: str):
    s = cell_name.lower()
    if '_hypoxia' in s or '_hypo_' in s or s.endswith('_hypo'):
        return 'Hypo'
    if '_normoxia' in s or '_norm_' in s or s.endswith('_norm'):
        return 'Norm'
    return None

meta_rows = []
for (tech, line, stage), df in matrices.items():
    for cell in df.columns:
        meta_rows.append({
            'cell_id': cell,
            'technology': tech,
            'cell_line': line,
            'stage': stage,
            'condition': extract_label(cell),
        })
meta = pd.DataFrame(meta_rows)

labels_by_cell = meta.drop_duplicates('cell_id').set_index('cell_id')['condition']

print('Cells per (technology, cell_line, condition), counted on the normalised stage only:')
counts = (meta[meta['stage'] == 'normalised']
          .groupby(['technology', 'cell_line', 'condition'])
          .size().unstack(fill_value=0))
counts

In [ ]:
matrices_log = {key: np.log1p(df.astype('float32')) for key, df in matrices.items()}

## 3. Exploratory data analysis

One shared exploratory pass for the whole report. We look at shape and type, the distribution of expression values, sparsity, outliers by interquartile range, the log transform, and duplicate rows.

### 3.1 Shapes, dtypes and descriptive statistics

In [ ]:
def quick_stats(df):
    vals = df.values.astype(float)
    return pd.Series({
        'n_genes': df.shape[0],
        'n_cells': df.shape[1],
        'dtype': str(df.dtypes.unique().tolist()),
        'min': vals.min(),
        'max': vals.max(),
        'mean': vals.mean(),
        'median': float(np.median(vals)),
        'std': vals.std(),
        'frac_zero': (vals == 0).mean(),
    })

stats_df = pd.DataFrame({f'{t}/{l}/{s}': quick_stats(df) for (t, l, s), df in matrices.items()}).T
stats_df

### 3.2 Expression-value distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, stage in zip(axes, ['unfiltered', 'filtered', 'normalised']):
    for (tech, line, s), df in matrices.items():
        if s != stage:
            continue
        sample = df.values.flatten()
        if sample.size > 200_000:
            sample = np.random.default_rng(SEED).choice(sample, 200_000, replace=False)
        ax.hist(np.log1p(sample), bins=80, alpha=0.5, density=True, label=f'{tech}/{line}')
    ax.set_title(f'{stage}, log1p(value)')
    ax.set_xlabel('log1p(expression)')
    ax.legend(fontsize=8)
axes[0].set_ylabel('density')
plt.tight_layout()
plt.show()

### 3.3 Sparsity

In [ ]:
sparsity_rows = []
for (tech, line, stage), df in matrices.items():
    vals = df.values
    sparsity_rows.append({
        'technology': tech, 'cell_line': line, 'stage': stage,
        'overall_frac_zero': float((vals == 0).mean()),
        'median_frac_zero_per_cell': float(np.median((vals == 0).mean(axis=0))),
        'median_frac_zero_per_gene': float(np.median((vals == 0).mean(axis=1))),
    })
sparsity_df = pd.DataFrame(sparsity_rows).sort_values(['technology', 'cell_line', 'stage']).reset_index(drop=True)
sparsity_df

### 3.4 Outliers and the interquartile range

In [ ]:
def iqr_flags(series: pd.Series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (series < lo) | (series > hi), (lo, hi)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, df) in zip(axes, [('MCF7 unfiltered', matrices[('SmartSeq', 'MCF7', 'unfiltered')]),
                                    ('HCC1806 unfiltered', matrices[('SmartSeq', 'HCC1806', 'unfiltered')])]):
    lib = df.sum(axis=0)
    detected = (df > 0).sum(axis=0)
    flagged, bounds = iqr_flags(lib)
    ax.scatter(lib, detected, c=np.where(flagged, 'red', 'steelblue'), s=12, alpha=0.7)
    ax.set_xlabel('library size (sum of counts)')
    ax.set_ylabel('detected genes')
    ax.set_title(f'{label}, {flagged.sum()}/{len(flagged)} outliers by library size')
plt.tight_layout()
plt.show()

### 3.5 Log transformation

In [ ]:
sample_df = matrices[('SmartSeq', 'MCF7', 'filtered')]
raw = sample_df.values.flatten()
logged = np.log1p(raw)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(raw[raw < np.quantile(raw, 0.99)], bins=80, color='grey')
axes[0].set_title('Filtered counts (clipped at 99th percentile)')
axes[0].set_xlabel('count')
axes[1].hist(logged, bins=80, color='steelblue')
axes[1].set_title('log1p(counts)')
axes[1].set_xlabel('log1p(count)')
plt.tight_layout()
plt.show()

### 3.6 Duplicate rows

In [ ]:
dup_rows = []
for (tech, line, stage), df in matrices.items():
    dup_rows.append({
        'technology': tech, 'cell_line': line, 'stage': stage,
        'dup_gene_rows': int(df.index.duplicated().sum()),
        'dup_cell_cols': int(df.columns.duplicated().sum()),
        'dup_gene_values': int(df.duplicated().sum()),
    })
pd.DataFrame(dup_rows)

## 4. Data structure

We move from per-value statistics to relationships between genes and between cells: a gene-gene correlation map, principal component analysis, and a heatmap of the top variable genes by condition. The PCA block also builds the per-technology structures (`per_tech_pca`, `shared_genes`) reused by the representation-learning and pooled-model sections.

### 4.1 Gene-gene correlation

In [ ]:
def prep_matrix(df):
    X = np.log1p(df.T.astype(float))
    return X

X_mcf7 = prep_matrix(matrices[('SmartSeq', 'MCF7', 'normalised')])
var_genes = X_mcf7.var(axis=0).nlargest(200).index
corr = X_mcf7[var_genes].corr()
g = sns.clustermap(corr, cmap='vlag', center=0, figsize=(8, 8),
                   xticklabels=False, yticklabels=False)
g.fig.suptitle('Gene-gene correlation, SmartSeq MCF7 (top 200 variable genes, log1p)', y=1.02)
plt.show()

### 4.2 Principal component analysis

PCA is run after log1p and per-gene standardisation, both across all four subsets and within each technology. The dominant axis of variation is the sequencing technology rather than the condition, a batch effect that the generalisation experiments later quantify.

In [ ]:
def _stack_tech(tech, mats):
    dfs   = {line: mats[(tech, line, 'normalised')] for line in ('MCF7', 'HCC1806')}
    genes = sorted(set.intersection(*(set(df.index) for df in dfs.values())))

    pieces = []
    for line, df in dfs.items():
        cells_x_genes = df.loc[genes].T.copy()
        cells_x_genes['cell_line']  = line
        cells_x_genes['technology'] = tech
        cells_x_genes['cell_id']    = cells_x_genes.index
        pieces.append(cells_x_genes)
    out = pd.concat(pieces, ignore_index=True)
    out['condition'] = labels_by_cell.reindex(out['cell_id']).values
    return out, genes

def _fit_pca(X, n_components=10):
    Xs  = StandardScaler().fit_transform(np.asarray(X, dtype=float))
    pca = PCA(n_components=n_components, random_state=SEED).fit(Xs)
    return Xs, pca, pca.transform(Xs)

def _scatter_mixed(ax, x, y, labels, s=8, alpha=0.5, seed=SEED):
    from matplotlib.lines import Line2D
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(x))
    x = np.asarray(x)[order]
    y = np.asarray(y)[order]
    labels = pd.Series(labels).fillna('NaN').astype(str).values[order]
    uniq = sorted(set(labels))
    cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    cmap = {l: cycle[i % len(cycle)] for i, l in enumerate(uniq)}
    ax.scatter(x, y, c=[cmap[l] for l in labels],
               s=s, alpha=alpha, edgecolors='none', linewidths=0)
    handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=cmap[l],
                      markersize=6, label=l) for l in uniq]
    ax.legend(handles=handles, fontsize=8)

per_tech_pca = {}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, tech in enumerate(['SmartSeq', 'DropSeq']):
    df_tech, gene_cols = _stack_tech(tech, matrices_log)
    Xs_tech, pca_tech, emb_tech = _fit_pca(df_tech[gene_cols].values)
    per_tech_pca[tech] = (df_tech, gene_cols, Xs_tech, pca_tech, emb_tech)
    print(f'{tech}: {len(gene_cols)} genes shared between MCF7 and HCC1806, {len(df_tech)} cells')

    s, alpha = (14, 0.65) if tech == 'SmartSeq' else (4, 0.25)
    ax_scree, ax_cond, ax_line = axes[row]
    ax_scree.plot(np.arange(1, 11), pca_tech.explained_variance_ratio_, marker='o')
    ax_scree.set_title(f'{tech} scree'); ax_scree.set_xlabel('PC'); ax_scree.set_ylabel('var ratio')

    _scatter_mixed(ax_cond, emb_tech[:, 0], emb_tech[:, 1],
                   df_tech['condition'].values, s=s, alpha=alpha)
    ax_cond.set_title(f'{tech} PC1 vs PC2 by condition')
    ax_cond.set_xlabel('PC1'); ax_cond.set_ylabel('PC2')

    _scatter_mixed(ax_line, emb_tech[:, 0], emb_tech[:, 1],
                   df_tech['cell_line'].values, s=s, alpha=alpha)
    ax_line.set_title(f'{tech} PC1 vs PC2 by cell line')
    ax_line.set_xlabel('PC1'); ax_line.set_ylabel('PC2')
plt.tight_layout(); plt.show()

normalised_subsets_log = {(t, l): matrices_log[(t, l, 'normalised')]
                        for t in ('SmartSeq', 'DropSeq') for l in ('MCF7', 'HCC1806')}
shared_genes = sorted(set.intersection(*(set(df.index) for df in normalised_subsets_log.values())))
print(f'Shared genes across all 4 normalised subsets: {len(shared_genes)}')

pieces = []
for (tech, line), df in normalised_subsets_log.items():
    sub = df.loc[shared_genes].T.copy()
    sub['technology'] = tech
    sub['cell_line']  = line
    sub['cell_id']    = sub.index
    pieces.append(sub)
combined = pd.concat(pieces, ignore_index=True)
combined['condition'] = labels_by_cell.reindex(combined['cell_id']).values

Xs, pca_combined, emb_combined = _fit_pca(combined[shared_genes].values)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(np.arange(1, 11), pca_combined.explained_variance_ratio_, marker='o')
axes[0].set_title('Combined scree'); axes[0].set_xlabel('PC'); axes[0].set_ylabel('var ratio')
_scatter_mixed(axes[1], emb_combined[:, 0], emb_combined[:, 1],
               combined['technology'].values, s=4, alpha=0.25)
axes[1].set_title('Combined PC1 vs PC2 by technology\n(batch-effect view)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for row, tech in enumerate(['SmartSeq', 'DropSeq']):
    for col, line in enumerate(['MCF7', 'HCC1806']):
        df = matrices_log[(tech, line, 'normalised')].T
        Xs_, _, emb_ = _fit_pca(df.values)
        cond = labels_by_cell.reindex(df.index).values
        s, alpha = (14, 0.65) if tech == 'SmartSeq' else (4, 0.25)
        _scatter_mixed(axes[row, col], emb_[:, 0], emb_[:, 1], cond, s=s, alpha=alpha)
        axes[row, col].set_title(f'{tech} / {line} - PC1 vs PC2 by condition')
        axes[row, col].set_xlabel('PC1'); axes[row, col].set_ylabel('PC2')
plt.tight_layout(); plt.show()

### 4.3 Heatmap of top variable genes by condition

In [ ]:
target = matrices[('SmartSeq', 'MCF7', 'normalised')]
labels = labels_by_cell.reindex(target.columns)
sub = target.loc[target.var(axis=1).nlargest(40).index]
col_order = labels.sort_values().index
sub = sub[col_order]
colour_map = labels.reindex(col_order).map({'Hypo': '#d62728', 'Norm': '#1f77b4'})
sns.clustermap(sub, col_cluster=False, z_score=0, cmap='vlag', center=0, figsize=(10, 7),
               col_colors=colour_map.values, yticklabels=True, xticklabels=False)
plt.show()

## 5. Supervised classification

Every classifier predicts Hypo versus Norm from gene expression and is scored with stratified five-fold cross-validation on ROC-AUC, accuracy and F1. The `StandardScaler` is fit on the training fold only, so no test information leaks. We share one feature builder: `Xys` holds the raw 3000-gene inputs per subset, and `Xys_fe` adds a single engineered Buffa hypoxia metagene score.

In [ ]:
def make_Xy(tech, line, add_buffa=False):
    df = matrices[(tech, line, 'normalised')]
    y = labels_by_cell.reindex(df.columns)
    mask = y.isin(['Hypo', 'Norm']).values
    X = df.T.values[mask]
    genes = list(df.index.values)
    if add_buffa:
        score = buffa_score(df)[mask].reshape(-1, 1)
        X = np.hstack([X, score])
        genes = genes + ['BUFFA_score']
    y = (y[mask] == 'Hypo').astype(int).values
    return X, y, np.array(genes)

def cv_score(clf, X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    scores = cross_validate(
        Pipeline([('sc', StandardScaler()), ('clf', clf)]),
        X, y, cv=skf,
        scoring=['roc_auc', 'accuracy', 'f1'],
        return_train_score=False,
    )
    return {k.replace('test_', ''): float(np.mean(v)) for k, v in scores.items() if k.startswith('test_')}

SUBSETS = [('SmartSeq', 'MCF7'), ('SmartSeq', 'HCC1806'), ('DropSeq', 'MCF7'), ('DropSeq', 'HCC1806')]
Xys = {s: make_Xy(*s) for s in SUBSETS}
for s, (X, y, _) in Xys.items():
    print(f'{s}: X={X.shape}, class balance={np.bincount(y)}')

### 5.1 Feature engineering, the Buffa hypoxia metagene score

We add exactly one engineered feature, the per-cell mean expression over a 45-gene offline reconstruction of the Buffa hypoxia signature, and feed it to the classifiers as an extra input column.

In [ ]:
BUFFA_HYPOXIA = ['ACOT7','ADM','ALDOA','ANKRD37','ANLN','BNIP3','CA9','CDKN3','CHCHD2','CORO1C',
    'CTPS1','DCBLD1','DDIT4','ENO1','ESRP1','FAM83B','GAPDH','GPI','HILPDA','HK2','KIF20A','LDHA',
    'LRRC42','MAD2L2','MCTS1','MIF','MRGBP','MRPL13','MRPS17','NDRG1','P4HA1','PGAM1','PGK1','PSMA7',
    'PSRC1','SHCBP1','SLC16A1','SLC2A1','SLC25A32','TPI1','TUBA1B','TUBB6','VEGFA','YKT6','AK4']

def buffa_score(df):
    present = [g for g in BUFFA_HYPOXIA if g in df.index]
    return df.loc[present].mean(axis=0).values

Xys_fe = {s: make_Xy(*s, add_buffa=True) for s in SUBSETS}

print('Buffa-metagene feature engineering')
for s in SUBSETS:
    df = matrices[(s[0], s[1], 'normalised')]
    n_present = sum(g in df.index for g in BUFFA_HYPOXIA)
    Xfe, yfe, _ = Xys_fe[s]
    score_only = Xfe[:, -1:]
    auc_score = cv_score(LogisticRegression(max_iter=1000, random_state=SEED), score_only, yfe)['roc_auc']
    print(f'  {s[0]:8s}/{s[1]:8s}: {n_present:2d}/{len(BUFFA_HYPOXIA)} Buffa genes present | '
          f'Buffa-score-only AUC = {auc_score:.3f}')

### 5.2 A shared cross-validation runner

Most classifiers are plain scikit-learn estimators, so we score them with one helper over the four subsets and collect the results into a single comparison table at the end.

In [ ]:
clf_results = {}

def run_clf(make_clf, name, data=None):
    data = Xys_fe if data is None else data
    res = {}
    for s, (X, y, g) in data.items():
        res[s] = cv_score(make_clf(), X, y)
        print(f'{name:14s} {s[0]:8s}/{s[1]:8s}: AUC={res[s]["roc_auc"]:.3f}  Acc={res[s]["accuracy"]:.3f}  F1={res[s]["f1"]:.3f}')
    clf_results[name] = res
    return res

### 5.3 LASSO, L1-regularised logistic regression

In [ ]:
LASSO_CS = np.logspace(-4, 4, 20)

def lasso_fit(X, y, genes):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    selected = np.zeros(X.shape[1], dtype=int)
    aucs = []
    for tr, te in skf.split(X, y):
        sc = StandardScaler().fit(X[tr])
        Xtr = sc.transform(X[tr]).astype('float32')
        Xte = sc.transform(X[te]).astype('float32')
        gs = GridSearchCV(
            LogisticRegression(penalty='l1', solver='saga', max_iter=3000, random_state=SEED),
            {'C': LASSO_CS}, cv=3, scoring='roc_auc', n_jobs=1,
        ).fit(Xtr, y[tr])
        clf = gs.best_estimator_
        prob = clf.predict_proba(Xte)[:, 1]
        aucs.append(roc_auc_score(y[te], prob))
        selected += (clf.coef_[0] != 0).astype(int)
    stability = pd.Series(selected / 5, index=genes).sort_values(ascending=False)
    return float(np.mean(aucs)), stability

lasso_results = {}
for subset, (X, y, g) in Xys_fe.items():
    auc, stab = lasso_fit(X, y, g)
    lasso_results[subset] = {'auc': auc, 'top_genes': stab.head(20)}
    print(f'LASSO {subset}: AUC={auc:.3f}, non-zero (>=80% folds) = {(stab >= 0.8).sum()}')

print('\nTop stable genes for SmartSeq/MCF7:')
print(lasso_results[('SmartSeq', 'MCF7')]['top_genes'].head(15))

### 5.4 Linear SVM

In [ ]:
svm_results = {}
for subset, (X, y, g) in Xys_fe.items():
    base = LinearSVC(C=1.0, random_state=SEED, max_iter=5000)
    clf = CalibratedClassifierCV(base, cv=3)
    svm_results[subset] = cv_score(clf, X, y)
    print(f'SVM {subset}: AUC={svm_results[subset]["roc_auc"]:.3f}  Acc={svm_results[subset]["accuracy"]:.3f}')

### 5.5 L2 Logistic Regression, Random Forest and Extra Trees

Three further estimators on the shared scaffold: a plain L2 logistic regression baseline, and two tree ensembles that can capture non-linear gene interactions the linear models cannot.

In [ ]:
logreg_results = run_clf(lambda: LogisticRegression(max_iter=2000, random_state=SEED), 'LogReg')
rf_results = run_clf(lambda: RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=1), 'RandomForest')
et_results = run_clf(lambda: ExtraTreesClassifier(n_estimators=200, random_state=SEED, n_jobs=1), 'ExtraTrees')

### 5.6 XGBoost on the raw genes

Gradient-boosted trees on the standardised gene inputs. The CPU `hist` tree method keeps the model portable across WSL and Windows. A richer representation-based XGBoost study follows in the next section.

In [ ]:
if XGB_OK:
    def make_xgb():
        return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                             subsample=0.9, colsample_bytree=0.9, tree_method='hist',
                             eval_metric='logloss', random_state=SEED, n_jobs=1)
    xgb_raw_results = run_clf(make_xgb, 'XGBoost')
else:
    xgb_raw_results = {}
    print('xgboost not installed, raw-gene XGBoost skipped.')

### 5.7 K Nearest Neighbours

KNN uses a per-technology pipeline: PCA followed by Euclidean distance for the deep SmartSeq data, and cosine distance for the sparse DropSeq data, with the neighbour count chosen by an inner cross-validation. The large DropSeq subsets are subsampled for the neighbour search.

In [ ]:
def make_knn(tech, k, n_comp=50):
    if tech == "SmartSeq":
        return Pipeline([
            ("sc", StandardScaler()),
            ("pca", PCA(n_components=n_comp, random_state=SEED)),
            ("knn", KNeighborsClassifier(n_neighbors=k, metric="euclidean"))
        ])
    elif tech == "DropSeq":
        return Pipeline([
            ("sc", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=k, metric="cosine"))
        ])

In [ ]:
def best_k(X, y, tech=None):
    k_grid = range(1, 52, 2) if tech == "DropSeq" else range(1, 32, 2)

    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    search = GridSearchCV(
        Pipeline([("sc", StandardScaler()),
                  ("knn", KNeighborsClassifier(metric="cosine"))]),
        param_grid={"knn__n_neighbors": list(k_grid)},
        cv=inner_cv, scoring="roc_auc", n_jobs=1
    )
    search.fit(X, y)
    return search.best_params_["knn__n_neighbors"]

In [ ]:
knn_results = {}
for (tech, line), (X, y, g) in Xys.items():
    Xs, ys = X, y
    if tech == 'DropSeq' and len(X) > 2000:
        idx = rng.choice(len(X), size=2000, replace=False)
        Xs, ys = X[idx], y[idx]
    k = best_k(Xs, ys, tech=tech)
    knn_results[(tech, line)] = cv_score(make_knn(tech, k), Xs, ys)
    print(f'KNN            {tech:8s}/{line:8s}: AUC={knn_results[(tech, line)]["roc_auc"]:.3f}  [k={k}]')
clf_results['KNN'] = knn_results

### 5.8 Multi-layer perceptron

In [ ]:
def make_mlp() -> Pipeline:
    return MLPClassifier(
        hidden_layer_sizes=(256, 64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=64,
        learning_rate_init=1e-3,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=SEED,
    )

In [ ]:
mlp_results = run_clf(make_mlp, 'MLP')

### 5.9 1D CNN

A small one-dimensional convolutional network that slides filters along genes ordered by variance. The gene order is arbitrary, so the CNN is an exploratory non-linear baseline rather than a biologically motivated model. It runs only when PyTorch is available.

In [ ]:
def train_cnn_fold(Xtr, ytr, Xte, yte, epochs=25, batch=256, lr=1e-3, seed=SEED):
    device = DEVICE
    use_amp = (device.type == 'cuda')

    class CNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(1, 16, kernel_size=7, padding=3), nn.ReLU(),
                nn.Conv1d(16, 16, kernel_size=7, padding=3), nn.ReLU(),
                nn.AdaptiveAvgPool1d(64), nn.Flatten(),
                nn.Linear(16 * 64, 32), nn.ReLU(),
                nn.Linear(32, 2))
        def forward(self, x):
            return self.net(x)

    torch.manual_seed(seed)
    model = CNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    Xtr_t = torch.tensor(np.asarray(Xtr), dtype=torch.float32, device=device).unsqueeze(1)
    ytr_t = torch.tensor(np.asarray(ytr), dtype=torch.long,    device=device)
    Xte_t = torch.tensor(np.asarray(Xte), dtype=torch.float32, device=device).unsqueeze(1)

    n = Xtr_t.shape[0]
    for _ in range(epochs):
        model.train()
        order = torch.randperm(n, device=device)
        for i in range(0, n, batch):
            idx = order[i:i+batch]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=use_amp):
                loss = loss_fn(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        prob = torch.softmax(model(Xte_t).float(), dim=1).cpu().numpy()[:, 1]
    return roc_auc_score(yte, prob)

cnn_results = {}
if TORCH_OK:
    for subset, (X, y, g) in Xys_fe.items():
        order = np.argsort(-X.var(axis=0))
        Xo = X[:, order]
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        aucs = []
        for tr, te in skf.split(Xo, y):
            sc = StandardScaler().fit(Xo[tr])
            Xtr = np.asarray(sc.transform(Xo[tr])).astype('float32')
            Xte = np.asarray(sc.transform(Xo[te])).astype('float32')
            aucs.append(train_cnn_fold(Xtr, y[tr], Xte, y[te]))
        cnn_results[subset] = {'roc_auc': float(np.mean(aucs))}
        print(f'1D CNN {subset}: AUC={cnn_results[subset]["roc_auc"]:.3f}')
else:
    print('PyTorch not installed - 1D CNN section skipped.')

### 5.10 Classifier comparison

One table, one row per subset, one column per classifier. Within-subset AUC is almost always high on this kind of data; the honest test is the cross-subset generalisation of Section 6.

In [ ]:
def _auc(d, s):
    if s not in d:
        return np.nan
    v = d[s]
    return v.get('roc_auc', v.get('auc', np.nan)) if isinstance(v, dict) else np.nan

comparison = {}
for s in SUBSETS:
    row = {'LASSO': lasso_results[s]['auc'], 'SVM': _auc(svm_results, s),
           'LogReg': _auc(logreg_results, s), 'RandomForest': _auc(rf_results, s),
           'ExtraTrees': _auc(et_results, s), 'KNN': _auc(knn_results, s),
           'MLP': _auc(mlp_results, s)}
    if XGB_OK:
        row['XGBoost'] = _auc(xgb_raw_results, s)
    if s in cnn_results:
        row['1D CNN'] = cnn_results[s]['roc_auc']
    comparison[f'{s[0]}/{s[1]}'] = row
comparison_df = pd.DataFrame(comparison).T
print(comparison_df.round(3).to_string())
comparison_df.style.background_gradient(cmap='Blues', axis=None).format('{:.3f}')

## 6. Representation learning, PCA features and an Autoencoder

Here we learn compact per-technology representations and feed them to XGBoost. Genes are intersected within a technology across the two cell lines, then standardised with the scaler fit on the training split only. We train one Autoencoder and one denoising Autoencoder per technology, compare their reconstruction against trivial and PCA baselines, and then compare XGBoost on the raw genes, on the PCA embedding and on the Autoencoder latent.

### 6.1 Per-technology train and validation split

In [ ]:
from sklearn.model_selection import train_test_split

ae_data = {}
for tech in ('SmartSeq', 'DropSeq'):
    df_tech, gene_cols, _, _, _ = per_tech_pca[tech]
    X = df_tech[gene_cols].values.astype('float32')

    strat = (df_tech['cell_line'].astype(str) + '|' +
             df_tech['condition'].fillna('NaN').astype(str)).values

    idx_tr, idx_va = train_test_split(
        np.arange(len(X)), test_size=0.2, random_state=SEED, stratify=strat
    )

    scaler = StandardScaler().fit(X[idx_tr])
    Xs_full = scaler.transform(X).astype('float32')

    ae_data[tech] = {
        'df_tech':  df_tech,
        'gene_cols': gene_cols,
        'scaler':   scaler,
        'idx_tr':   idx_tr,
        'idx_va':   idx_va,
        'Xs_full':  Xs_full,
        'Xtr':      Xs_full[idx_tr],
        'Xva':      Xs_full[idx_va],
    }
    print(f'{tech}: n_in={X.shape[1]} genes, train={len(idx_tr)} cells, val={len(idx_va)} cells')

### 6.2 Autoencoder and denoising Autoencoder

A linear-bottleneck Autoencoder with a Bernoulli-mask denoising variant, trained per technology with early stopping. The clean input is always the reconstruction target.

In [ ]:
class AE(nn.Module):
    def __init__(self, n_in, latent=32, hidden=256, depth=1, dropout=0.0, use_bn=False):
        super().__init__()

        def block(in_dim, out_dim):
            layers = [nn.Linear(in_dim, out_dim)]
            if use_bn:
                layers.append(nn.BatchNorm1d(out_dim))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            return layers

        enc_layers, in_dim = [], n_in
        for _ in range(depth):
            enc_layers += block(in_dim, hidden)
            in_dim = hidden
        enc_layers.append(nn.Linear(in_dim, latent))
        self.enc = nn.Sequential(*enc_layers)

        dec_layers, in_dim = [], latent
        for _ in range(depth):
            dec_layers += block(in_dim, hidden)
            in_dim = hidden
        dec_layers.append(nn.Linear(in_dim, n_in))
        self.dec = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), z

def train_ae(Xtr, Xva, X_full, latent=32, hidden=256, depth=1, dropout=0.0, use_bn=False,
             denoising=False, noise_type='bernoulli', noise_level=0.3,
             epochs=80, batch=256, lr=1e-3, weight_decay=0.0,
             patience=10, seed=SEED, verbose=False):
    torch.manual_seed(seed)
    device = DEVICE
    n_in = Xtr.shape[1]

    model = AE(n_in, latent=latent, hidden=hidden, depth=depth,
               dropout=dropout, use_bn=use_bn).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=device)
    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=device)
    n = Xtr_t.shape[0]

    history = {'train': [], 'val': []}
    best_val = float('inf')
    best_state = None
    bad_epochs = 0

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        ep_loss = 0.0
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            xb = Xtr_t[idx]

            if denoising:
                if noise_type == 'bernoulli':
                    mask = (torch.rand_like(xb) > noise_level).float()
                    xb_in = xb * mask
                elif noise_type == 'gaussian':
                    xb_in = xb + noise_level * torch.randn_like(xb)
                else:
                    raise ValueError(f'unknown noise_type: {noise_type}')
            else:
                xb_in = xb

            opt.zero_grad(set_to_none=True)
            recon, _ = model(xb_in)
            loss = loss_fn(recon, xb)
            loss.backward()
            opt.step()
            ep_loss += float(loss.detach()) * xb.size(0)
        ep_loss /= n

        model.eval()
        with torch.no_grad():
            recon_va, _ = model(Xva_t)
            val_loss = float(loss_fn(recon_va, Xva_t))

        history['train'].append(ep_loss)
        history['val'].append(val_loss)

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

        if verbose and (ep % 10 == 0 or bad_epochs >= patience):
            print(f'    ep {ep:3d}  train {ep_loss:.4f}  val {val_loss:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        Xfull_t = torch.tensor(X_full, dtype=torch.float32, device=device)
        _, Z = model(Xfull_t)
        Z = Z.detach().cpu().numpy()

    return model, history, Z

ae_models, dae_models = {}, {}
for tech in ('SmartSeq', 'DropSeq'):
    d = ae_data[tech]
    print(f'== {tech} ==')

    print('  AE...')
    model, history, Z = train_ae(d['Xtr'], d['Xva'], d['Xs_full'])
    ae_models[tech] = {'model': model, 'history': history, 'Z': Z,
                       'epochs_trained': len(history['train']),
                       'best_val': min(history['val'])}
    print(f"    {len(history['train'])} epochs, best val MSE = {min(history['val']):.4f}")

    print('  DAE (Bernoulli mask, p=0.3)...')
    model, history, Z = train_ae(d['Xtr'], d['Xva'], d['Xs_full'],
                                 denoising=True, noise_type='bernoulli', noise_level=0.3)
    dae_models[tech] = {'model': model, 'history': history, 'Z': Z,
                        'epochs_trained': len(history['train']),
                        'best_val': min(history['val'])}
    print(f"    {len(history['train'])} epochs, best val MSE = {min(history['val']):.4f}")

### 6.3 Reconstruction baselines and training curves

In [ ]:
baselines = []
for tech in ('SmartSeq', 'DropSeq'):
    d = ae_data[tech]
    pca32 = PCA(n_components=32, random_state=SEED).fit(d['Xtr'])
    Xva_recon = pca32.inverse_transform(pca32.transform(d['Xva']))
    baselines.append({
        'tech': tech,
        'n_in': d['Xtr'].shape[1],
        'compression': f"{d['Xtr'].shape[1] / 32:.1f}x",
        'trivial val MSE': float(np.mean(d['Xva'] ** 2)),
        'PCA(32) val MSE': float(np.mean((d['Xva'] - Xva_recon) ** 2)),
        'PCA(32) cum var explained': float(pca32.explained_variance_ratio_.sum()),
        'AE val MSE (current)': ae_models[tech]['best_val'],
    })
pd.DataFrame(baselines)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for col, tech in enumerate(['SmartSeq', 'DropSeq']):
    for row, (label, models) in enumerate([('AE', ae_models), ('DAE', dae_models)]):
        ax = axes[row, col]
        h = models[tech]['history']
        ax.plot(h['train'], label='train', alpha=0.85)
        ax.plot(h['val'], label='val', alpha=0.85)
        best_ep = int(np.argmin(h['val']))
        ax.axvline(best_ep, color='grey', linestyle=':', alpha=0.6, label=f'best val at ep {best_ep}')
        ax.set_title(f'{tech} {label}  (best val MSE = {min(h["val"]):.3f})')
        ax.set_xlabel('epoch'); ax.set_ylabel('MSE'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 6.4 Choosing the Autoencoder per technology

We pick the better of the Autoencoder and denoising Autoencoder per technology by how separable their latent space is for hypoxia, measured with a logistic-regression probe on the validation cells.

In [ ]:
def sep_auc(Z, df_tech, idx_tr, idx_va):
    cond = df_tech['condition'].values
    n = len(cond)
    known = pd.notna(cond)
    y = (cond == 'Hypo').astype(int)
    mtr = np.zeros(n, bool); mtr[idx_tr] = True; mtr &= known
    mva = np.zeros(n, bool); mva[idx_va] = True; mva &= known
    if mtr.sum() < 10 or mva.sum() < 10 or len(np.unique(y[mva])) < 2:
        return float('nan')
    clf = LogisticRegression(max_iter=2000, random_state=SEED).fit(Z[mtr], y[mtr])
    return roc_auc_score(y[mva], clf.predict_proba(Z[mva])[:, 1])

ae_winner = {}
for tech in ('SmartSeq', 'DropSeq'):
    d = ae_data[tech]
    cand = {'AE': ae_models[tech]['Z'], 'DAE': dae_models[tech]['Z']}
    scored = {k: sep_auc(Z, d['df_tech'], d['idx_tr'], d['idx_va']) for k, Z in cand.items()}
    best = max(scored, key=lambda k: (scored[k] if scored[k] == scored[k] else -1))
    ae_winner[tech] = {'name': best, 'auc': scored[best], 'Z': cand[best]}
    print(f'{tech:8s} winner = {best}  (latent val AUC = {scored[best]:.3f})')

### 6.5 XGBoost on raw, PCA and Autoencoder representations

Three feature views per technology fed to the same gradient-boosted tree trainer, with per-round learning curves.

In [ ]:
xgb_reps = {}
for tech in ('SmartSeq', 'DropSeq'):
    df_tech, gene_cols, Xs_tech, _, emb_tech = per_tech_pca[tech]
    d = ae_data[tech]
    reps = {
        'raw': df_tech[gene_cols].values.astype('float32'),
        'pca': np.asarray(emb_tech, dtype='float32'),
        'ae': np.asarray(ae_winner[tech]['Z'], dtype='float32'),
    }
    cond = df_tech['condition'].values
    known = pd.notna(cond)
    y = (cond == 'Hypo').astype(int)
    n = len(df_tech)
    m_tr = np.zeros(n, bool); m_tr[d['idx_tr']] = True; m_tr &= known
    m_va = np.zeros(n, bool); m_va[d['idx_va']] = True; m_va &= known
    xgb_reps[tech] = {'reps': reps, 'y': y, 'm_tr': m_tr, 'm_va': m_va}
    print(f"{tech:8s} | train={m_tr.sum():5d} val={m_va.sum():5d} | "
          f"raw={reps['raw'].shape[1]}d pca={reps['pca'].shape[1]}d ae={reps['ae'].shape[1]}d")

In [ ]:
from xgboost import XGBClassifier

def train_xgboost(X_tr, y_tr, X_va, y_va, *,
                  n_estimators=400, max_depth=4, learning_rate=0.1,
                  subsample=0.9, colsample_bytree=0.9, min_child_weight=1.0,
                  reg_lambda=1.0, reg_alpha=0.0, early_stopping_rounds=30,
                  seed=SEED, **extra_params):
    params = dict(
        n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
        subsample=subsample, colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight, reg_lambda=reg_lambda, reg_alpha=reg_alpha,
        objective='binary:logistic', eval_metric=['error', 'auc'],
        early_stopping_rounds=early_stopping_rounds, tree_method='hist',
        n_jobs=1, random_state=seed, **extra_params,
    )
    model = XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_va, y_va)], verbose=False)

    res = model.evals_result()
    tr, va = res['validation_0'], res['validation_1']
    history = {
        'train': {'acc': [1.0 - e for e in tr['error']], 'auc': list(tr['auc'])},
        'val':   {'acc': [1.0 - e for e in va['error']], 'auc': list(va['auc'])},
        'best_iteration': int(model.best_iteration),
        'params': params,
    }
    bi = history['best_iteration']
    history['best_val_acc'] = history['val']['acc'][bi]
    history['best_val_auc'] = history['val']['auc'][bi]
    return model, history

In [ ]:
if XGB_OK:
    xgb_models = {}
    for tech in ('SmartSeq', 'DropSeq'):
        R = xgb_reps[tech]
        for rep_name, X in R['reps'].items():
            model, history = train_xgboost(X[R['m_tr']], R['y'][R['m_tr']],
                                           X[R['m_va']], R['y'][R['m_va']])
            xgb_models[(tech, rep_name)] = {'model': model, 'history': history}
            print(f"{tech:8s} {rep_name:4s} | best round {history['best_iteration'] + 1:3d} | "
                  f"val acc {history['best_val_acc']:.3f} | val auc {history['best_val_auc']:.3f}")
    xgb_summary = pd.DataFrame([
        {'tech': t, 'representation': r, 'n_features': xgb_reps[t]['reps'][r].shape[1],
         'best_round': m['history']['best_iteration'] + 1,
         'val_acc': m['history']['best_val_acc'], 'val_auc': m['history']['best_val_auc']}
        for (t, r), m in xgb_models.items()]).sort_values(['tech', 'representation']).reset_index(drop=True)
    print(xgb_summary.round(3).to_string())
else:
    xgb_models = {}
    print('xgboost not installed, representation XGBoost skipped.')

## 7. Graph Neural Networks

A complementary view: build a k-nearest-neighbour graph of cells (each node is a cell, edges join similar cells) and a graph of genes (each node is a gene), then train Graph Convolutional Network and GraphSAGE node classifiers to predict hypoxia. Graph models need torch-geometric, so this section runs only when `PYG_OK` is true. A torch-geometric-free Random Forest with co-expression-graph-weighted gene importance always runs as a fallback.

### 7.1 Adapting the shared data to a graph dataset

In [ ]:
def gnn_expr_labels(tech, line):
    df = matrices[(tech, line, 'normalised')]
    lab = labels_by_cell.reindex(df.columns)
    return df, lab

datasets = {f'{t}_{l}': gnn_expr_labels(t, l)
            for t in ('SmartSeq', 'DropSeq') for l in ('MCF7', 'HCC1806')}

loaded = {}
for line in ('MCF7', 'HCC1806'):
    df, lab = gnn_expr_labels('SmartSeq', line)
    Xg = df.T.copy()
    yg = lab.loc[Xg.index].astype(str).str.contains('Hyp', case=False, na=False).astype(int).values
    loaded[line] = {'X': Xg, 'y': yg}

GENE_SETS = {'BUFFA_HYPOXIA': set(BUFFA_HYPOXIA)}
DATASET_NAME = 'SmartSeq_MCF7'
print('GNN graph dataset:', DATASET_NAME, '| torch_geometric available:', PYG_OK)

### 7.2 Graph construction and training utilities

In [ ]:
def build_cell_graph(expr, labels, k=10):
    X, y = prepare_xy(expr, labels)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    adjacency = kneighbors_graph(
        X_scaled,
        n_neighbors=k,
        mode="connectivity",
        include_self=False
    )
    adjacency = adjacency.maximum(adjacency.T)

    edge_index_np = np.vstack(adjacency.nonzero())

    data = Data(
        x=torch.tensor(X_scaled, dtype=torch.float32),
        edge_index=torch.tensor(edge_index_np, dtype=torch.long),
        y=torch.tensor(y.values, dtype=torch.long)
    )

    data.sample_names = list(X.index)
    data.gene_names = list(X.columns)

    return data, X, y, adjacency

cell_data, cell_X, cell_y, cell_adj = build_cell_graph(expr, labels, k=10)

print(cell_data)
print("Number of nodes:", cell_data.num_nodes)
print("Number of edges:", cell_data.num_edges)
print("Number of node features:", cell_data.num_features)

In [ ]:
def add_node_masks(data, y, train_size=0.7, val_size=0.15):
    indices = np.arange(data.num_nodes)

    train_idx, temp_idx = train_test_split(
        indices,
        train_size=train_size,
        random_state=SEED,
        stratify=y
    )

    val_relative = val_size / (1 - train_size)

    val_idx, test_idx = train_test_split(
        temp_idx,
        train_size=val_relative,
        random_state=SEED,
        stratify=y.iloc[temp_idx]
    )

    data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    data.val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    data.test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

    data.train_mask[train_idx] = True
    data.val_mask[val_idx] = True
    data.test_mask[test_idx] = True

    data.train_idx = train_idx
    data.val_idx = val_idx
    data.test_idx = test_idx

    return data

cell_data = add_node_masks(cell_data, cell_y)
cell_data = cell_data.to(DEVICE)

print("Train nodes:", int(cell_data.train_mask.sum()))
print("Val nodes:", int(cell_data.val_mask.sum()))
print("Test nodes:", int(cell_data.test_mask.sum()))

In [ ]:
def make_gene_labels(expr, labels):
    condition = labels.loc[expr.columns]
    hyp_cols = condition[condition.astype(str).str.contains("Hyp", case=False, na=False)].index
    norm_cols = condition[condition.astype(str).str.contains("Norm", case=False, na=False)].index

    hyp_mean = expr[hyp_cols].mean(axis=1)
    norm_mean = expr[norm_cols].mean(axis=1)
    gene_label = (hyp_mean > norm_mean).astype(int)

    diff_df = pd.DataFrame({
        "mean_hypoxia": hyp_mean,
        "mean_normoxia": norm_mean,
        "diff": hyp_mean - norm_mean,
        "gene_label": gene_label
    })
    return gene_label, diff_df

gene_labels, gene_diff_df = make_gene_labels(expr, labels)
print(gene_labels.value_counts())
display(gene_diff_df.head())

In [ ]:
def build_gene_graph(expr, labels, k=10, max_genes=1000):
    gene_label, diff_df = make_gene_labels(expr, labels)

    if expr.shape[0] > max_genes:
        selected_genes = expr.var(axis=1).sort_values(ascending=False).head(max_genes).index
        expr_sub = expr.loc[selected_genes]
        gene_label = gene_label.loc[selected_genes]
        diff_df = diff_df.loc[selected_genes]
    else:
        expr_sub = expr.copy()

    X = expr_sub.copy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    adjacency = kneighbors_graph(
        X_scaled,
        n_neighbors=k,
        mode="connectivity",
        include_self=False
    )
    adjacency = adjacency.maximum(adjacency.T)

    edge_index_np = np.vstack(adjacency.nonzero())

    data = Data(
        x=torch.tensor(X_scaled, dtype=torch.float32),
        edge_index=torch.tensor(edge_index_np, dtype=torch.long),
        y=torch.tensor(gene_label.values, dtype=torch.long)
    )
    data.gene_names = list(expr_sub.index)

    return data, expr_sub, gene_label, diff_df, adjacency

gene_data, gene_expr_sub, gene_y, gene_diff_sub, gene_adj = build_gene_graph(expr, labels, k=10, max_genes=1000)

print(gene_data)
print("Number of gene nodes:", gene_data.num_nodes)
print("Number of edges:", gene_data.num_edges)
print("Number of features per gene:", gene_data.num_features)

In [ ]:
def prepare_xy(expr, labels):
    X = expr.T.copy()
    y = labels.loc[X.index].astype(str).str.contains('Hyp', case=False, na=False).astype(int)
    return X, y

### 7.3 GCN and GraphSAGE node classifiers

In [ ]:
class GCNNodeClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.out = GCNConv(hidden_dim, output_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.out(x, edge_index)
        return x

class GraphSAGENodeClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.out = SAGEConv(hidden_dim, output_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.out(x, edge_index)
        return x

In [ ]:
def train_gnn_node_classifier(model, data, epochs=150, lr=1e-3, weight_decay=1e-4):
    model = model.to(DEVICE)
    data = data.to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])

        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index)
            pred = logits.argmax(dim=1)

            train_acc = accuracy_score(
                data.y[data.train_mask].cpu().numpy(),
                pred[data.train_mask].cpu().numpy()
            )
            val_acc = accuracy_score(
                data.y[data.val_mask].cpu().numpy(),
                pred[data.val_mask].cpu().numpy()
            )
            val_f1 = f1_score(
                data.y[data.val_mask].cpu().numpy(),
                pred[data.val_mask].cpu().numpy()
            )

        history.append({
            "epoch": epoch,
            "loss": float(loss.item()),
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "val_f1": val_f1
        })

        if epoch == 1 or epoch % 25 == 0:
            print(
                f"Epoch {epoch:03d} | loss {loss.item():.4f} | "
                f"train acc {train_acc:.3f} | val acc {val_acc:.3f}"
            )

    return pd.DataFrame(history)

def evaluate_gnn_node_classifier(model, data, split="test"):
    model = model.to(DEVICE)
    data = data.to(DEVICE)

    model.eval()
    mask = getattr(data, f"{split}_mask")

    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        pred = logits.argmax(dim=1)

    y_true = data.y[mask].cpu().numpy()
    y_pred = pred[mask].cpu().numpy()

    print(classification_report(y_true, y_pred, target_names=["Normoxia", "Hypoxia"]))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }, pd.DataFrame({"true": y_true, "pred": y_pred})

In [ ]:
gnn_results_table = pd.DataFrame()
if PYG_OK:
    expr, labels = datasets[DATASET_NAME]

    cell_data, cell_X, cell_y, cell_adj = build_cell_graph(expr, labels, k=10)
    cell_data = add_node_masks(cell_data, cell_y)
    tr, va, te = cell_data.train_idx, cell_data.val_idx, cell_data.test_idx
    scaler = StandardScaler()
    Xsc = np.zeros_like(cell_X.values, dtype=np.float32)
    Xsc[tr] = scaler.fit_transform(cell_X.iloc[tr])
    Xsc[va] = scaler.transform(cell_X.iloc[va])
    Xsc[te] = scaler.transform(cell_X.iloc[te])
    cell_data.x = torch.tensor(Xsc, dtype=torch.float32).to(DEVICE)

    gcn_cell = GCNNodeClassifier(input_dim=cell_data.num_features, hidden_dim=64, output_dim=2)
    train_gnn_node_classifier(gcn_cell, cell_data, epochs=150)
    gcn_cell_metrics, _ = evaluate_gnn_node_classifier(gcn_cell, cell_data, split='test')

    sage_cell = GraphSAGENodeClassifier(input_dim=cell_data.num_features, hidden_dim=64, output_dim=2)
    train_gnn_node_classifier(sage_cell, cell_data, epochs=150)
    sage_cell_metrics, _ = evaluate_gnn_node_classifier(sage_cell, cell_data, split='test')

    gene_data, gene_expr_sub, gene_y, gene_diff, gene_adj = build_gene_graph(expr, labels, k=10, max_genes=1000)
    gene_data = add_node_masks(gene_data, pd.Series(gene_data.y.numpy()))
    gene_data = gene_data.to(DEVICE)

    gcn_gene = GCNNodeClassifier(input_dim=gene_data.num_features, hidden_dim=64, output_dim=2)
    train_gnn_node_classifier(gcn_gene, gene_data, epochs=150)
    gcn_gene_metrics, _ = evaluate_gnn_node_classifier(gcn_gene, gene_data, split='test')

    sage_gene = GraphSAGENodeClassifier(input_dim=gene_data.num_features, hidden_dim=64, output_dim=2)
    train_gnn_node_classifier(sage_gene, gene_data, epochs=150)
    sage_gene_metrics, _ = evaluate_gnn_node_classifier(sage_gene, gene_data, split='test')

    gnn_results_table = pd.DataFrame([
        {'graph': 'cell', 'model': 'GCN', **gcn_cell_metrics},
        {'graph': 'cell', 'model': 'GraphSAGE', **sage_cell_metrics},
        {'graph': 'gene', 'model': 'GCN', **gcn_gene_metrics},
        {'graph': 'gene', 'model': 'GraphSAGE', **sage_gene_metrics},
    ])
    print(gnn_results_table.round(3).to_string(index=False))
else:
    print('torch_geometric not installed, GCN and GraphSAGE skipped.')

### 7.4 Graph-aware gene importance without torch-geometric

A Random Forest provides node importance, weighted by each gene's degree in a co-expression graph (absolute correlation thresholded at a high quantile). This runs everywhere and recovers a graph-derived hypoxia gene ranking.

In [ ]:
def graph_weighted_importance(cell_line, top_variable=500, corr_quantile=0.995):
    X = loaded[cell_line]['X']
    y = loaded[cell_line]['y']
    genes = X.var(axis=0).sort_values(ascending=False).head(min(top_variable, X.shape[1])).index.tolist()
    Xs = X[genes]
    Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=0.25, random_state=SEED, stratify=y)
    rf = RandomForestClassifier(n_estimators=150, random_state=SEED, class_weight='balanced', n_jobs=1)
    rf.fit(Xtr, ytr)
    proba = rf.predict_proba(Xte)[:, 1]
    metrics = {'cell_line': cell_line, 'accuracy': accuracy_score(yte, rf.predict(Xte)),
               'f1': f1_score(yte, rf.predict(Xte)), 'roc_auc': roc_auc_score(yte, proba)}
    imp = pd.DataFrame({'gene': genes, 'node_importance': rf.feature_importances_})
    corr = np.corrcoef(StandardScaler().fit_transform(Xs).T)
    np.fill_diagonal(corr, 0.0)
    abs_corr = np.abs(corr)
    adj = abs_corr >= np.quantile(abs_corr, corr_quantile)
    imp['coexpression_degree'] = adj.sum(axis=0)
    imp['graph_weighted_importance'] = imp['node_importance'] * (1 + np.log1p(imp['coexpression_degree']))
    imp['hypoxia_related'] = imp['gene'].str.upper().isin({g.upper() for g in BUFFA_HYPOXIA})
    return imp.sort_values('graph_weighted_importance', ascending=False), metrics

graph_importance, graph_rf_metrics = {}, []
for line in loaded:
    imp, met = graph_weighted_importance(line)
    graph_importance[line] = imp
    graph_rf_metrics.append(met)
    top = imp.head(12)['gene'].tolist()
    print(f'{line:8s} RF graph AUC={met["roc_auc"]:.3f} | top graph genes: {", ".join(top)}')
pd.DataFrame(graph_rf_metrics).round(3)

## 8. Unsupervised views, K-Means and t-SNE

Two label-free looks at the data. K-Means clusters each subset into two groups and we compare those clusters to the true condition. t-SNE gives a non-linear two-dimensional embedding coloured by condition.

### 8.1 K-Means clustering versus condition

In [ ]:
def kmeans_pipeline():
    return Pipeline([('scaler', StandardScaler()),
                     ('kmeans', KMeans(n_clusters=2, random_state=SEED, n_init=20))])

def align_clusters(clusters, y_true):
    maj = np.bincount(y_true[clusters == 0].astype(np.int64)).argmax()
    aligned = np.where(clusters == 0, maj, 1 - maj)
    if accuracy_score(y_true, aligned) < 0.5:
        aligned = 1 - aligned
    return aligned

print('K-Means cluster agreement with the true condition (ARI and aligned accuracy):')
for (tech, line), (X, y, g) in Xys.items():
    clusters = kmeans_pipeline().fit_predict(X)
    ari = adjusted_rand_score(y, clusters)
    acc = accuracy_score(y, align_clusters(clusters, y))
    print(f'  {tech:8s}/{line:8s}: ARI={ari:.3f}  aligned acc={acc:.3f}')

### 8.2 t-SNE embedding by condition

In [ ]:
def tsne_plot(tech, line):
    X, y, g = Xys[(tech, line)]
    Xs = StandardScaler().fit_transform(np.log1p(X.astype(float)))
    if Xs.shape[0] > 4000:
        idx = rng.choice(Xs.shape[0], 4000, replace=False)
        Xs, ys = Xs[idx], y[idx]
    else:
        ys = y
    perp = min(30, max(5, (Xs.shape[0] - 1) // 3))
    emb = TSNE(n_components=2, perplexity=perp, random_state=SEED, init='pca', learning_rate='auto').fit_transform(Xs)
    return emb, ys

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (tech, line) in zip(axes, SUBSETS):
    emb, ys = tsne_plot(tech, line)
    for val, name, col in [(1, 'Hypo', '#d62728'), (0, 'Norm', '#1f77b4')]:
        m = ys == val
        ax.scatter(emb[m, 0], emb[m, 1], s=6, alpha=0.6, label=name, color=col)
    ax.set_title(f'{tech}/{line} t-SNE'); ax.legend(fontsize=8)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout(); plt.show()

## 9. Cross-cell-line and cross-technology generalisation

The most important experiment. Within each subset the classifiers reach very high AUC, but that may only mean the model memorised the batch. We now train on one subset and evaluate on a different one, restricting both matrices to their shared genes, using LASSO as the interpretable baseline.

In [ ]:
def cross_eval(src, dst, model_fn):
    df_s = matrices[(src[0], src[1], 'normalised')]
    df_d = matrices[(dst[0], dst[1], 'normalised')]
    shared = sorted(set(df_s.index) & set(df_d.index))
    y_s = labels_by_cell.loc[df_s.columns]
    y_d = labels_by_cell.loc[df_d.columns]
    ms = y_s.isin(['Hypo', 'Norm']).values
    md = y_d.isin(['Hypo', 'Norm']).values
    Xs = df_s.loc[shared].T.values[ms]
    Xd = df_d.loc[shared].T.values[md]
    ys = (y_s[ms] == 'Hypo').astype(int).values
    yd = (y_d[md] == 'Hypo').astype(int).values
    sc = StandardScaler().fit(Xs)
    clf = model_fn().fit(sc.transform(Xs).astype('float32'), ys)
    Xd_t = sc.transform(Xd).astype('float32')
    if hasattr(clf, 'predict_proba'):
        prob = clf.predict_proba(Xd_t)[:, 1]
    else:
        prob = clf.decision_function(Xd_t)
    return roc_auc_score(yd, prob), len(shared)

def make_lasso_gs():
    return GridSearchCV(
        LogisticRegression(penalty='l1', solver='saga', max_iter=2000, random_state=SEED),
        {'C': np.logspace(-3, 3, 10)}, cv=3, scoring='roc_auc', n_jobs=1,
    )

models = {
    'LASSO': make_lasso_gs,
}

rows = []
for name, mfn in models.items():
    for src in SUBSETS:
        for dst in SUBSETS:
            if src == dst:
                continue
            auc, n_shared = cross_eval(src, dst, mfn)
            rows.append({'model': name,
                         'train': f'{src[0]}/{src[1]}',
                         'test':  f'{dst[0]}/{dst[1]}',
                         'n_shared_genes': n_shared,
                         'auc': auc})
gen_df = pd.DataFrame(rows)
for name in models:
    pivot = gen_df[gen_df.model == name].pivot(index='train', columns='test', values='auc')
    print(f'\n== {name} cross-subset AUC ==')
    print(pivot.round(3))

## 10. Feature importance and hypoxia biology

A near-perfect within-subset AUC is not on its own evidence of biology. This section finds the genes each model uses, tests whether they recover the known hypoxia programme, and checks for a core set shared across subsets.

### 10.1 Feature importance, consensus of LASSO and SVM

In [ ]:
LASSO_C_IMP = 0.1

def lasso_importance(X, y, genes, C=LASSO_C_IMP):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    sel = np.zeros(X.shape[1]); coef = np.zeros(X.shape[1])
    for tr, _ in skf.split(X, y):
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(penalty='l1', solver='saga', C=C, max_iter=4000,
                                 random_state=SEED).fit(sc.transform(X[tr]).astype('float32'), y[tr])
        c = clf.coef_[0]; sel += (np.abs(c) > 1e-6); coef += np.abs(c)
    return pd.Series(sel / 5, index=genes), pd.Series(coef / 5, index=genes)

def svm_importance(X, y, genes):
    sc = StandardScaler().fit(X)
    svc = LinearSVC(C=1.0, random_state=SEED, max_iter=5000).fit(sc.transform(X).astype('float32'), y)
    return pd.Series(np.abs(svc.coef_[0]), index=genes)

IMPORTANCE = {}
for s in SUBSETS:
    X, y, g = Xys[s]
    stab, coef = lasso_importance(X, y, g)
    IMPORTANCE[s] = {'lasso_stab': stab, 'lasso_coef': coef, 'svm': svm_importance(X, y, g)}
    print(f'{s[0]:8s}/{s[1]:8s}: LASSO stable(>=0.6)={int((stab >= 0.6).sum()):3d} | '
          f'top SVM={IMPORTANCE[s]["svm"].idxmax():10s}')

In [ ]:
def consensus_rank(imp):
    R = pd.DataFrame({'LASSO': imp['lasso_coef'].rank(pct=True),
                      'SVM':   imp['svm'].rank(pct=True)})
    R['consensus'] = R[['LASSO', 'SVM']].mean(axis=1)
    return R.sort_values('consensus', ascending=False)

CONSENSUS = {s: consensus_rank(IMPORTANCE[s]) for s in SUBSETS}

STAB_THR = 0.6
TOP_K    = 50
PRED = {}
for s in SUBSETS:
    stab = IMPORTANCE[s]['lasso_stab']
    stable = stab[stab >= STAB_THR].index
    if len(stable) < 10:
        stable = CONSENSUS[s].head(100).index
    ranked = CONSENSUS[s].loc[CONSENSUS[s].index.isin(stable)].index.tolist()
    PRED[s] = ranked[:TOP_K]
    print(f'{s[0]:8s}/{s[1]:8s}: {int((stab >= STAB_THR).sum()):4d} LASSO-stable -> '
          f'top-{len(PRED[s])} signature | {", ".join(PRED[s][:8])}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, s in zip(axes.ravel(), SUBSETS):
    top = CONSENSUS[s].head(15)['consensus'].iloc[::-1]
    ax.barh(range(len(top)), top.values, color='steelblue')
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index, fontsize=8)
    ax.set_title(f'{s[0]}/{s[1]} - top-15 consensus genes', fontsize=10)
    ax.set_xlabel('mean percentile rank (LASSO|coef|, SVM)', fontsize=8)
plt.tight_layout(); plt.show()

top_union = sorted(set().union(*[set(CONSENSUS[s].head(12).index) for s in SUBSETS]))
H = pd.DataFrame({f'{s[0][:5]}/{s[1]}': CONSENSUS[s]['consensus'].reindex(top_union) for s in SUBSETS})
fig, ax = plt.subplots(figsize=(7, max(4, len(top_union) * 0.26)))
sns.heatmap(H, cmap='magma', vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'consensus rank'})
ax.set_title('Consensus importance across subsets (union of each subset top-12)')
plt.tight_layout(); plt.show()

### 10.2 Reference hypoxia signatures

In [ ]:
HIF_TARGETS = {
    'glycolysis':         ['SLC2A1','SLC2A3','HK1','HK2','PFKL','PFKFB3','ALDOA','ALDOC','GAPDH','PGK1','PGAM1','ENO1','TPI1','LDHA','PDK1'],
    'angiogenesis':       ['VEGFA','VEGFC','ANGPT2','ANGPTL4','FLT1','KDR','NRP1','EDN1'],
    'pH_regulation':      ['CA9','CA12','SLC9A1','SLC16A1','SLC16A3'],
    'survival_autophagy': ['BNIP3','BNIP3L','HILPDA','DDIT4','BCL2L1'],
    'erythro_iron':       ['EPO','TFRC','TF'],
    'ecm_invasion':       ['LOX','LOXL2','MMP2','MMP9','CXCR4','P4HA1','P4HA2','PLOD1','PLOD2'],
    'cell_cycle_stress':  ['CDKN1A','NDRG1','DDIT3','ADM'],
}
HIF_TARGETS_FLAT = sorted({g for v in HIF_TARGETS.values() for g in v})

BUFFA_HYPOXIA = ['ACOT7','ADM','ALDOA','ANKRD37','ANLN','BNIP3','CA9','CDKN3','CHCHD2','CORO1C',
    'CTPS1','DCBLD1','DDIT4','ENO1','ESRP1','FAM83B','GAPDH','GPI','HILPDA','HK2','KIF20A','LDHA',
    'LRRC42','MAD2L2','MCTS1','MIF','MRGBP','MRPL13','MRPS17','NDRG1','P4HA1','PGAM1','PGK1','PSMA7',
    'PSRC1','SHCBP1','SLC16A1','SLC2A1','SLC25A32','TPI1','TUBA1B','TUBB6','VEGFA','YKT6','AK4']
HALLMARK_HYPOXIA_CORE = ['VEGFA','SLC2A1','LDHA','PGK1','PDK1','ALDOA','ALDOC','ENO1','ENO2','GAPDH',
    'PGAM2','TPI1','HK1','HK2','PFKL','PFKFB3','GPI','PGM1','PGM2','ADM','BNIP3L','DDIT4','NDRG1',
    'CA12','P4HA1','P4HA2','PLOD1','PLOD2','LOX','VLDLR','ANGPTL4','SERPINE1','GYS1','PPP1R3C','STC1',
    'STC2','ERO1A','HILPDA','DDIT3','CITED2','EGLN3','CDKN1A','CDKN1B','GBE1','IGFBP3','TGFB3','PIM1','MIF']
WINTER_HYPOXIA = ['SLC2A1','CA9','VEGFA','LDHA','PGK1','ENO1','ALDOA','BNIP3','NDRG1','ADM','P4HA1',
    'PGAM1','TPI1','HK2','SLC16A1','MRPS17','MIF','GPI','ANKRD37','HILPDA','DDIT4','KCTD11','FAM162A']

HYPOXIA_SIGS = {'HIF_targets_curated':   HIF_TARGETS_FLAT,
                'BUFFA_HYPOXIA':         sorted(set(BUFFA_HYPOXIA)),
                'HALLMARK_HYPOXIA_core': sorted(set(HALLMARK_HYPOXIA_CORE)),
                'WINTER_HYPOXIA':        sorted(set(WINTER_HYPOXIA))}

BACKGROUND = sorted(set().union(*[set(g) for (_, _, g) in Xys.values()]))
print(f'Measured gene universe (union across subsets): {len(BACKGROUND)} HGNC symbols\n')
print('Signature coverage in the measured background:')
for name, genes in HYPOXIA_SIGS.items():
    present = sorted(set(genes) & set(BACKGROUND))
    print(f'  {name:22s} {len(present):3d}/{len(set(genes)):3d} measured  e.g. {", ".join(present[:8])}')

### 10.3 Do the predictive genes recover the hypoxia programme

In [ ]:
from scipy.stats import hypergeom

def overlap_test(gene_list, signature, background):
    bg = set(background); gl = set(gene_list) & bg; sig = set(signature) & bg
    k = len(gl & sig); M, n, N = len(bg), len(sig), len(gl)
    p = hypergeom.sf(k - 1, M, n, N) if (k > 0 and n > 0 and N > 0) else 1.0
    return k, n, N, p, sorted(gl & sig)

def bh_fdr(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    ranked = np.minimum.accumulate((p[order] * m / (np.arange(m) + 1))[::-1])[::-1]
    out = np.empty(m); out[order] = np.clip(ranked, 0, 1); return out

rows = []
for s in SUBSETS:
    bg = Xys[s][2]
    for sig_name, sig_genes in HYPOXIA_SIGS.items():
        k, nsig, nhit, p, shared = overlap_test(PRED[s], sig_genes, bg)
        rows.append({'subset': f'{s[0][:5]}/{s[1]}', 'signature': sig_name, 'overlap': k,
                     'sig_in_bg': nsig, 'n_pred': nhit, 'p': p, 'shared': ', '.join(shared[:10])})
overlap_df = pd.DataFrame(rows)
overlap_df['FDR'] = bh_fdr(overlap_df['p'].values)
fmt = lambda v: (f'{v:.2e}' if v < 0.01 else f'{v:.3f}')
print(overlap_df[['subset','signature','overlap','sig_in_bg','n_pred','p','FDR']]
      .to_string(index=False, formatters={'p': fmt, 'FDR': fmt}))
print('\nNamed overlapping genes:')
for _, r in overlap_df.iterrows():
    if r['shared']:
        print(f'  {r["subset"]:13s} {r["signature"]:22s} {r["shared"]}')

### 10.4 Pathway-level breakdown

In [ ]:
from collections import Counter

cat_rows = []
for s in SUBSETS:
    preds = set(PRED[s]); row = {'subset': f'{s[0][:5]}/{s[1]}'}
    for cat, genes in HIF_TARGETS.items():
        row[cat] = len(preds & set(genes))
    cat_rows.append(row)
cat_df = pd.DataFrame(cat_rows).set_index('subset')
print('Predictive genes per HIF programme category:')
print(cat_df.to_string())

fig, ax = plt.subplots(figsize=(9, 3.4))
sns.heatmap(cat_df, annot=True, fmt='d', cmap='Greens', ax=ax, cbar_kws={'label': '# predictive genes'})
ax.set_title('HIF-programme coverage by predictive genes'); plt.tight_layout(); plt.show()

hit_counter = Counter()
for s in SUBSETS:
    hit_counter.update(set(PRED[s]) & set(HIF_TARGETS_FLAT))
recur = pd.Series(hit_counter).sort_values(ascending=False)
print('\nCanonical HIF targets that are predictive, by # of subsets:')
print(recur.to_string() if len(recur) else '  (none)')

### 10.5 A core gene set shared across subsets

In [ ]:
import itertools
sets = {s: set(PRED[s]) for s in SUBSETS}
labels = [f'{t[:5]}/{l}' for (t, l) in SUBSETS]
J = pd.DataFrame(index=labels, columns=labels, dtype=float)
for (la, sa), (lb, sb) in itertools.product(zip(labels, sets.values()), repeat=2):
    J.loc[la, lb] = len(sa & sb) / len(sa | sb) if (sa | sb) else 0.0
fig, ax = plt.subplots(figsize=(5.2, 4.2))
sns.heatmap(J.astype(float), annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, ax=ax)
ax.set_title('Jaccard overlap of predictive-gene sets'); plt.tight_layout(); plt.show()

core_all   = set.intersection(*sets.values())
core_smart = sets[('SmartSeq','MCF7')] & sets[('SmartSeq','HCC1806')]
core_drop  = sets[('DropSeq','MCF7')]  & sets[('DropSeq','HCC1806')]
print(f'CORE across all 4 subsets ({len(core_all)}): {sorted(core_all)}')
print(f'CORE SmartSeq (MCF7 & HCC1806) ({len(core_smart)}): {sorted(core_smart)}')
print(f'CORE DropSeq  (MCF7 & HCC1806) ({len(core_drop)}): {sorted(core_drop)}')
print('\nCore-set (all-4) overlap with hypoxia signatures:')
for name, genes in HYPOXIA_SIGS.items():
    inter = core_all & set(genes)
    print(f'  {name:22s} {sorted(inter) if inter else "-"}')

### 10.6 SHAP attributions on the gene-level XGBoost

When the SHAP library and XGBoost are available, TreeSHAP gives signed per-gene attributions on the validation cells, a model-agnostic cross-check of the LASSO and SVM consensus.

In [ ]:
if SHAP_OK and XGB_OK and xgb_models:
    shap_xgb_raw = {}
    for tech in ('SmartSeq', 'DropSeq'):
        model = xgb_models[(tech, 'raw')]['model']
        gene_cols = per_tech_pca[tech][1]
        R = xgb_reps[tech]
        X_va = pd.DataFrame(R['reps']['raw'][R['m_va']], columns=gene_cols)
        sv = shap.TreeExplainer(model)(X_va)
        vals = sv.values if sv.values.ndim == 2 else sv.values[..., 1]
        shap_xgb_raw[tech] = vals
        top = pd.Series(np.abs(vals).mean(0), index=gene_cols).sort_values(ascending=False)
        print(f'{tech}: top genes by mean absolute SHAP: ' + ', '.join(top.head(12).index))
        shap.summary_plot(vals, X_va, max_display=15, show=False)
        plt.title(f'{tech} XGBoost raw-gene SHAP, positive pushes toward hypoxia')
        plt.tight_layout(); plt.show()
else:
    print('shap or xgboost not available, SHAP attribution skipped.')

## 11. Combined-cell-line pooled model

We pool data across cell lines and test whether a single classifier generalises: first within a technology holding out one cell line, then across all four subsets holding out one subset at a time. The scaler is fit on the training fold only.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut

def build_pool(subsets, gene_space):
    Xs, ys, grp = [], [], []
    for (tech, line) in subsets:
        df = matrices[(tech, line, 'normalised')]
        yb = labels_by_cell.reindex(df.columns)
        m = yb.isin(['Hypo', 'Norm']).values
        Xs.append(df.loc[gene_space].T.values[m])
        ys.append((yb[m] == 'Hypo').astype(int).values)
        grp += [f'{tech}/{line}'] * int(m.sum())
    return np.vstack(Xs), np.concatenate(ys), np.array(grp)

def logo_eval(subsets, gene_space, model_fn, group_by='subset'):
    X, y, grp = build_pool(subsets, gene_space)
    if group_by == 'cell_line':
        grp = np.array([g.split('/')[1] for g in grp])
    out = []
    for tr, te in LeaveOneGroupOut().split(X, y, grp):
        sc = StandardScaler().fit(X[tr])
        clf = model_fn().fit(sc.transform(X[tr]).astype('float32'), y[tr])
        prob = (clf.predict_proba(sc.transform(X[te]).astype('float32'))[:, 1]
                if hasattr(clf, 'predict_proba')
                else clf.decision_function(sc.transform(X[te]).astype('float32')))
        out.append({'held_out': grp[te][0], 'n_test': len(te), 'auc': roc_auc_score(y[te], prob)})
    return pd.DataFrame(out)

pool_models = {
    'LASSO': lambda: LogisticRegression(penalty='l1', solver='saga', C=0.1, max_iter=3000, random_state=SEED),
}

print('=== (a) Pooled WITHIN technology, leave-one-cell-line-out ===')
for tech in ['SmartSeq', 'DropSeq']:
    subs = [(tech, 'MCF7'), (tech, 'HCC1806')]
    gspace = sorted(set(matrices[(tech, 'MCF7', 'normalised')].index)
                    & set(matrices[(tech, 'HCC1806', 'normalised')].index))
    for name, fn in pool_models.items():
        r = logo_eval(subs, gspace, fn, group_by='cell_line')
        print(f'  {tech:8s} {name:5s} mean AUC={r.auc.mean():.3f} | '
              + ' '.join(f'{x.held_out}={x.auc:.3f}' for _, x in r.iterrows()))

print('\n=== (b) Pooled across ALL 4 subsets, leave-one-subset-out ===')
pooled_logo = {}
for name, fn in pool_models.items():
    r = logo_eval(SUBSETS, shared_genes, fn, group_by='subset')
    pooled_logo[name] = r
    print(f'  {name:5s} mean AUC={r.auc.mean():.3f}')
    print(r.to_string(index=False))

In [ ]:
within = pd.Series({f'{s[0]}/{s[1]}': lasso_results[s]['auc'] for s in SUBSETS}, name='within_subset_LASSO')
held = pooled_logo['LASSO'].set_index('held_out')['auc'].rename('pooled_heldout_LASSO')
cmp = pd.concat([within, held], axis=1).round(3)
print('Within-subset vs pooled-held-out AUC (LASSO):')
print(cmp.to_string())

X_all, y_all, _ = build_pool(SUBSETS, shared_genes)
sc_all = StandardScaler().fit(X_all)
pooled_clf = LogisticRegression(penalty='l1', solver='saga', C=0.1, max_iter=3000,
                                random_state=SEED).fit(sc_all.transform(X_all).astype('float32'), y_all)
frames = []
for subset, path in test_paths.items():
    if not Path(path).exists():
        print(f'[skip] {Path(path).name} missing'); continue
    tdf = load_expr(path).reindex(shared_genes).fillna(0.0)
    prob = pooled_clf.predict_proba(sc_all.transform(tdf.T.values).astype('float32'))[:, 1]
    frames.append(pd.DataFrame({'cell_id': tdf.columns.astype(str), 'technology': subset[0],
                                'cell_line': subset[1], 'prob_hypo_pooled': prob,
                                'pred_pooled': np.where(prob >= 0.5, 'Hypo', 'Norm')}))
if frames:
    pooled_predictions = pd.concat(frames, ignore_index=True)
    pooled_predictions.to_csv(OUT_DIR / 'predictions_pooled.tsv', sep='\t', index=False)
    print(f'\nWrote {len(pooled_predictions)} pooled predictions to predictions_pooled.tsv')
    base = pd.read_csv(OUT_DIR / 'predictions.tsv', sep='\t')
    base['cell_id'] = base['cell_id'].astype(str)
    merged = base.merge(pooled_predictions, on=['cell_id', 'technology', 'cell_line'], how='inner')
    if len(merged):
        agree = (merged['pred'] == merged['pred_pooled']).mean()
        print(f'Agreement pooled vs per-subset predictions: {agree:.1%} of {len(merged)} cells')

## 12. Independence of the two cell lines, MCF7 versus HCC1806

A focused statistical study of whether the two cell lines are independent, in three complementary senses, each within a single technology. Part A asks whether the two lines are distinguishable at all. Part B asks whether they share co-expression structure. Part C asks whether the hypoxia response itself is shared. Parts A and B need the dcor and scikit-bio libraries and are skipped if those are absent; Part C always runs.

### 12.1 Distinguishability, classifier two-sample test, MMD and energy distance

In [ ]:
if DCOR_OK:
    import dcor
    from scipy.stats import norm
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold, cross_val_score

    def c2st(X, y, n_perm=50, max_n=6000, seed=SEED):
        rng = np.random.default_rng(seed)
        if len(y) > max_n:
            idx = rng.choice(len(y), max_n, replace=False)
            X, y = X[idx], y[idx]
        clf = LogisticRegression(max_iter=1000)
        cv = StratifiedKFold(5, shuffle=True, random_state=seed)
        obs = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc').mean()
        null = np.array([cross_val_score(clf, X, rng.permutation(y), cv=cv,
                                         scoring='roc_auc').mean() for _ in range(n_perm)])
        return float(obs), float((1 + np.sum(null >= obs)) / (1 + n_perm))

    def mmd_linear(X, Y, gamma=None, seed=SEED):
        rng = np.random.default_rng(seed)
        X, Y = np.asarray(X, float), np.asarray(Y, float)
        m = min(len(X), len(Y))
        X = X[rng.permutation(len(X))[:m]]
        Y = Y[rng.permutation(len(Y))[:m]]
        n2 = m // 2
        X1, X2, Y1, Y2 = X[:n2], X[n2:2*n2], Y[:n2], Y[n2:2*n2]
        if gamma is None:
            Z = np.vstack([X, Y]); sub = Z[rng.permutation(len(Z))[:min(500, len(Z))]]
            sq = np.sum(sub**2, axis=1)
            d2 = np.maximum(sq[:, None] + sq[None, :] - 2 * sub @ sub.T, 0.0)
            med = np.median(d2[np.triu_indices_from(d2, 1)])
            gamma = 1.0 / med if med > 0 else 1.0
        rbf = lambda a, b: np.exp(-gamma * np.sum((a - b) ** 2, axis=1))
        h = rbf(X1, X2) + rbf(Y1, Y2) - rbf(X1, Y2) - rbf(X2, Y1)
        stat, s = h.mean(), h.std(ddof=1)
        z = np.sqrt(n2) * stat / s if s > 0 else np.inf
        return float(stat), float(norm.sf(z))

    def energy_test(X, Y, max_n=2500, num_resamples=99, seed=SEED):
        rng = np.random.default_rng(seed)
        if len(X) > max_n: X = X[rng.choice(len(X), max_n, replace=False)]
        if len(Y) > max_n: Y = Y[rng.choice(len(Y), max_n, replace=False)]
        res = dcor.homogeneity.energy_test(X, Y, num_resamples=num_resamples, random_state=seed)
        pval = getattr(res, 'pvalue', None)
        if pval is None: pval = getattr(res, 'p_value', np.nan)
        return float(res.statistic), float(pval)

    rows_A, disc_genes = [], {}
    for tech in ('SmartSeq', 'DropSeq'):
        df_tech, gene_cols, Xs_tech, _, _ = per_tech_pca[tech]
        y = (df_tech['cell_line'].values == 'HCC1806').astype(int)
        Xm, Xh = Xs_tech[y == 0], Xs_tech[y == 1]

        auc, p_c2st = c2st(Xs_tech, y)
        mmd, p_mmd  = mmd_linear(Xm, Xh)
        edist, p_e  = energy_test(Xm, Xh)
        rows_A.append({'tech': tech, 'n_MCF7': int((y == 0).sum()), 'n_HCC': int((y == 1).sum()),
                       'C2ST_AUC': auc, 'C2ST_p': p_c2st, 'MMD_lin': mmd, 'MMD_p': p_mmd,
                       'energy_dist': edist, 'energy_p': p_e})

        coef = LogisticRegression(max_iter=1000).fit(Xs_tech, y).coef_[0]
        disc_genes[tech] = pd.Series(np.abs(coef), index=gene_cols).sort_values(ascending=False)
        print(f"{tech}: top line-discriminating genes -> " + ", ".join(disc_genes[tech].head(12).index))

    print("\n(High C2ST AUC + tiny MMD/energy p-values = the two lines are strongly "
          "distinguishable, i.e. NOT independent at the distribution level - as expected.)")
    A_results = pd.DataFrame(rows_A)
    A_results.round(4)
else:
    print('dcor not installed, independence study Part A skipped.')

### 12.2 Shared generative structure, correlation of correlations, RV, Mantel and principal angles

In [ ]:
if SKBIO_OK:
    from scipy.stats import spearmanr
    from scipy.linalg import subspace_angles
    from skbio.stats.distance import DistanceMatrix, mantel

    def rv_coefficient(S1, S2):
        return float(np.trace(S1 @ S2) / np.sqrt(np.trace(S1 @ S1) * np.trace(S2 @ S2)))

    K_GENES, K_SUB = 300, 10
    rows_B = {}
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, tech in zip(axes, ('SmartSeq', 'DropSeq')):
        df_tech, gene_cols, _, _, _ = per_tech_pca[tech]
        X = df_tech[gene_cols]
        line = df_tech['cell_line'].values

        topg = X.var(axis=0).nlargest(min(K_GENES, X.shape[1])).index
        Xm = X.loc[line == 'MCF7',    topg].values
        Xh = X.loc[line == 'HCC1806', topg].values

        Cm, Ch = np.nan_to_num(np.corrcoef(Xm.T)), np.nan_to_num(np.corrcoef(Xh.T))
        iu = np.triu_indices(len(topg), k=1)

        rho = spearmanr(Cm[iu], Ch[iu])[0]
        rv  = rv_coefficient(Cm, Ch)
        Dm, Dh = 1 - Cm, 1 - Ch
        for D in (Dm, Dh):
            D[:] = (D + D.T) / 2; np.fill_diagonal(D, 0.0)
        mant_r, mant_p, _ = mantel(DistanceMatrix(Dm), DistanceMatrix(Dh),
                                   method='spearman', permutations=999)
        Vm = PCA(K_SUB, random_state=SEED).fit(Xm).components_.T
        Vh = PCA(K_SUB, random_state=SEED).fit(Xh).components_.T
        mean_canon = float(np.mean(np.cos(subspace_angles(Vm, Vh))))

        rows_B[tech] = {'spearman_corr': rho, 'RV': rv, 'mantel_r': mant_r,
                        'mantel_p': mant_p, 'mean_canonical_corr': mean_canon}
        print(f"{tech}: Spearman(corr)={rho:.3f}  RV={rv:.3f}  "
              f"Mantel r={mant_r:.3f} (p={mant_p:.3g})  meanCanonCorr={mean_canon:.3f}")

        ax.hexbin(Cm[iu], Ch[iu], gridsize=60, cmap='viridis', bins='log')
        ax.plot([-1, 1], [-1, 1], 'r-', lw=1)
        ax.set_title(f'{tech}: gene-gene corr (top {len(topg)} var genes)\n'
                     f'Spearman={rho:.2f}, RV={rv:.2f}, Mantel r={mant_r:.2f}')
        ax.set_xlabel('MCF7 correlation'); ax.set_ylabel('HCC1806 correlation')
    plt.tight_layout(); plt.show()

    print("\n(Moderate-to-high values = the two lines share co-expression structure "
          "even though their absolute expression distributions differ.)")
    B_results = pd.DataFrame(rows_B).T
    B_results.round(4)
else:
    print('scikit-bio not installed, independence study Part B skipped.')

### 12.3 Functional independence, hypoxia logFC concordance and cross-line transfer

In [ ]:
from scipy.stats import spearmanr, hypergeom
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

def hypoxia_logfc(Xdf, cond):
    return Xdf[cond == 'Hypo'].mean(axis=0) - Xdf[cond == 'Norm'].mean(axis=0)

def rrho_grid(r1, r2, genes, step=20, k_max=300):
    o1 = np.array(genes)[np.argsort(-r1.values)]
    o2 = np.array(genes)[np.argsort(-r2.values)]
    ns = list(range(step, min(k_max, len(genes)) + 1, step))
    N = len(genes)
    M = np.zeros((len(ns), len(ns)))
    for a, na in enumerate(ns):
        s1 = set(o1[:na])
        for b, nb in enumerate(ns):
            ov = len(s1 & set(o2[:nb]))
            M[a, b] = -np.log10(max(hypergeom.sf(ov - 1, N, na, nb), 1e-300))
    return ns, M

def _jac(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b)

rows_C = []
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for row, tech in enumerate(('SmartSeq', 'DropSeq')):
    df_tech, gene_cols, _, _, _ = per_tech_pca[tech]
    X = df_tech[gene_cols]
    line, cond = df_tech['cell_line'].values, df_tech['condition'].values
    lab = pd.notna(cond)
    m_mask = lab & (line == 'MCF7'); h_mask = lab & (line == 'HCC1806')

    fc_m = hypoxia_logfc(X[m_mask], cond[m_mask])
    fc_h = hypoxia_logfc(X[h_mask], cond[h_mask])
    rho  = spearmanr(fc_m, fc_h)[0]
    j_up = _jac(fc_m.nlargest(100).index,  fc_h.nlargest(100).index)
    j_dn = _jac(fc_m.nsmallest(100).index, fc_h.nsmallest(100).index)

    y = (cond == 'Hypo').astype(int)
    def fit_eval(tr, te):
        sc  = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=1000).fit(sc.transform(X[tr]), y[tr])
        return roc_auc_score(y[te], clf.predict_proba(sc.transform(X[te]))[:, 1])
    def cv_auc(mask):
        pipe = Pipeline([('sc', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=1000))])
        return cross_val_score(pipe, X[mask], y[mask], scoring='roc_auc',
                               cv=StratifiedKFold(5, shuffle=True, random_state=SEED)).mean()
    auc_m2h, auc_h2m = fit_eval(m_mask, h_mask), fit_eval(h_mask, m_mask)
    ceil_m, ceil_h   = cv_auc(m_mask), cv_auc(h_mask)

    rows_C.append({'tech': tech, 'spearman_logFC': rho, 'jaccard_up': j_up,
                   'jaccard_dn': j_dn, 'MCF7_cv_AUC': ceil_m, 'HCC_cv_AUC': ceil_h,
                   'MCF7->HCC_AUC': auc_m2h, 'HCC->MCF7_AUC': auc_h2m})
    print(f"{tech}: Spearman(logFC)={rho:.3f}  Jacc(up)={j_up:.2f}  Jacc(dn)={j_dn:.2f} | "
          f"ceiling M={ceil_m:.3f} H={ceil_h:.3f} | transfer M->H={auc_m2h:.3f}  H->M={auc_h2m:.3f}")

    ax = axes[row, 0]
    ax.scatter(fc_m, fc_h, s=6, alpha=0.3)
    lim = float(max(np.abs(fc_m).max(), np.abs(fc_h).max()))
    ax.axhline(0, color='grey', lw=.5); ax.axvline(0, color='grey', lw=.5)
    ax.plot([-lim, lim], [-lim, lim], 'r-', lw=1)
    ax.set_title(f'{tech}: hypoxia logFC  (Spearman={rho:.2f})')
    ax.set_xlabel('MCF7 logFC'); ax.set_ylabel('HCC1806 logFC')

    ns, M = rrho_grid(fc_m, fc_h, list(gene_cols))
    ax = axes[row, 1]
    im = ax.imshow(M, origin='lower', aspect='auto', cmap='magma',
                   extent=[ns[0], ns[-1], ns[0], ns[-1]])
    ax.set_title(f'{tech}: RRHO (-log10 p, up-regulated)')
    ax.set_xlabel('HCC1806 top-n'); ax.set_ylabel('MCF7 top-n')
    fig.colorbar(im, ax=ax, fraction=0.046)

    ax = axes[row, 2]
    ax.bar(['M cv', 'H cv', 'M->H', 'H->M'], [ceil_m, ceil_h, auc_m2h, auc_h2m],
           color=['steelblue', 'steelblue', 'indianred', 'indianred'])
    ax.axhline(0.5, color='grey', ls='-'); ax.set_ylim(0, 1)
    ax.set_title(f'{tech}: within-line ceiling vs cross-line transfer')
    ax.set_ylabel('AUC')
plt.tight_layout(); plt.show()

print("\n(Positive logFC concordance + cross-line transfer AUC well above 0.5 = the "
      "hypoxia program is largely SHARED, i.e. functionally NOT independent, even "
      "though baseline expression is line-specific.)")
C_results = pd.DataFrame(rows_C)
C_results.round(4)

## 13. Prediction on the unseen anonymised test data

The course provides anonymised test files whose columns hide the labels. We train the LASSO baseline on the raw genes of each subset and write predictions for grading.

In [ ]:
test_paths = {
    ('SmartSeq', 'MCF7'):    SMART / 'MCF7_SmartS_Filtered_Normalised_3000_Data_test_anonim.txt',
    ('SmartSeq', 'HCC1806'): SMART / 'HCC1806_SmartS_Filtered_Normalised_3000_Data_test_anonim.txt',
    ('DropSeq',  'MCF7'):    DROP  / 'MCF7_Filtered_Normalised_3000_Data_test_anonim.txt',
    ('DropSeq',  'HCC1806'): DROP  / 'HCC1806_Filtered_Normalised_3000_Data_test_anonim.csv',
}

pred_frames = []
for subset, path in test_paths.items():
    if not path.exists():
        print(f'[skip] {path.name} missing')
        continue
    X_tr, y_tr, g_tr = Xys[subset]
    sc = StandardScaler().fit(X_tr)
    Xtr_t = sc.transform(X_tr).astype('float32')
    clf = GridSearchCV(
        LogisticRegression(penalty='l1', solver='saga', max_iter=3000, random_state=SEED),
        {'C': np.logspace(-3, 3, 10)}, cv=3, scoring='roc_auc', n_jobs=1,
    ).fit(Xtr_t, y_tr).best_estimator_
    test_df = load_expr(path)
    shared = [g for g in g_tr if g in test_df.index]
    if len(shared) < len(g_tr):
        missing = [g for g in g_tr if g not in test_df.index]
        fill = pd.DataFrame(0.0, index=missing, columns=test_df.columns)
        test_df = pd.concat([test_df, fill]).loc[g_tr]
    else:
        test_df = test_df.loc[g_tr]
    Xte = sc.transform(test_df.T.values).astype('float32')
    prob = clf.predict_proba(Xte)[:, 1]
    pred = pd.DataFrame({'cell_id': test_df.columns,
                         'technology': subset[0], 'cell_line': subset[1],
                         'prob_hypo': prob,
                         'pred': np.where(prob >= 0.5, 'Hypo', 'Norm')})
    pred_frames.append(pred)

if pred_frames:
    predictions = pd.concat(pred_frames, ignore_index=True)
    predictions.to_csv(OUT_DIR / 'predictions.tsv', sep='\t', index=False)
    print(f'Wrote {len(predictions)} predictions to {(OUT_DIR / "predictions.tsv").resolve()}')
    predictions.head()
else:
    print('No test files found on disk.')

## 14. Conclusions

Within every cell-line and technology subset the classifiers separate hypoxia from normoxia with high accuracy, and the linear models, the tree ensembles, the MLP and the graph networks largely agree. High within-subset accuracy alone is not evidence of biology, so the weight of the conclusion rests on generalisation and on the genes themselves.

Transfer splits sharply by technology. Within SmartSeq a signature learned on one cell line works on the other; within DropSeq transfer is weaker and asymmetric; and a single model pooled across all four subsets collapses toward chance, exactly as the technology-dominated PCA predicts. The independence study makes the same point precisely: the two cell lines are clearly distinguishable in absolute expression, yet they share co-expression structure and, most importantly, a largely shared hypoxia response, so cross-line transfer stays well above chance.

The predictive genes are genuinely hypoxic. PGK1 recurs across subsets, glycolysis dominates the pathway breakdown, and the overlap with published hypoxia signatures is significant after multiple-testing correction. SHAP on the gene-level XGBoost and the graph-weighted Random Forest importance point at the same programme. The autoencoder and graph models add capacity and a second perspective without changing the headline: the hypoxia signal is real and learnable, the technology batch effect is real too, and a deployable cross-platform classifier would need explicit batch correction before pooling.

The expression matrices are confidential and never leave the local machine. This notebook contains only aggregate results, gene names and figures. GPU floating point is not bit-for-bit reproducible, so metrics can vary slightly between runs. The published hypoxia signatures are reconstructed offline and should be confirmed against the canonical databases before any final submission.